In [ ]:
from ddsp import DDSP, AudioDataset
from ddsp.synths import ComplexSineSynth
from lightning import Trainer

from IPython.display import Audio, display
from torch.utils.data import DataLoader, Subset

import torch

torch.set_float32_matmul_precision('medium')
torch.set_default_tensor_type('torch.cuda.FloatTensor')
torch.set_default_device('cuda')

import os

experiment_root = '/home/btadeusz/code/ddsp_vae/experiments/complex_sine_synth/'
training_root = os.path.join(experiment_root, 'training')

In [ ]:
# Dataset parameters
chunk_duration = 2.0
sampling_rate = 44100
n_signal = chunk_duration * sampling_rate
batch_size = 8

# Model parameters
latent_size = num_params = 4
max_freq = 22050
n_sines = 10

# Training parameters
warmup_start = 300
warmup_end = 500
beta = 1.0
max_epochs = 1000
learning_rate = 1e-4


In [ ]:
def get_dataset_split(dataset_path, validation_split=0.2):
  """
  Splits the dataset into training and validation sets.
  """
  generator = torch.Generator(device='cuda')

  dataset_A = AudioDataset(dataset_path=dataset_path, n_signal=n_signal)
  total_len = len(dataset_A)

  val_len = int(validation_split * total_len)  # 20% for validation
  indices = torch.randperm(total_len, generator=generator)

  val_indices = indices[:val_len]
  train_indices = indices[val_len:]

  train_set = Subset(dataset_A, train_indices)
  val_set = Subset(dataset_A, val_indices)

  train_loader = DataLoader(train_set, batch_size, shuffle=True, num_workers=0, generator=generator)
  val_loader = DataLoader(val_set, batch_size, shuffle=False, num_workers=0, generator=generator)

  return train_loader, val_loader


In [ ]:
complex_sines =  ComplexSineSynth.to_config(
  n_sines=n_sines,
  fs=sampling_rate,
)

ddsp = DDSP(
  synth_configs=[complex_sines],
  fs=sampling_rate,
  latent_size=latent_size,
  num_params=num_params,
  learning_rate=learning_rate,
  perceptual_loss_weight=0,
  plateau_patience=200,
).to('cuda')

trainer = Trainer(
  max_epochs=1000,
  accelerator='cuda',
  precision=16
)

train_loader, val_loader = get_dataset_split('/mnt/mariadata/datasets/syntex/DS_BasicFM_1.1/audio')
trainer.fit(
    model=ddsp,
    train_dataloaders=train_loader,
    val_dataloaders=val_loader,
)

# Experimenting with minimal example

In [ ]:
from ddsp.synths import complex_oscillator
import math
import torch
torch.set_float32_matmul_precision('medium')
torch.set_default_tensor_type('torch.cuda.FloatTensor')
torch.set_default_device('cuda')

from IPython.display import Audio, display

def rad2hz(rad, fs):
  return rad * fs / (2 * math.pi)

def hz2rad(hz, fs):
  return hz * 2 * math.pi / fs


fs = 44100
nyquist_freq = fs // 2
dur = 2.0 # seconds

N = int(fs * dur)  # number of samples
n = torch.arange(N) # time vector

# torch.random.manual_seed(1000)
starting_freq_hz = torch.rand(1) * nyquist_freq
# Convert frequency to radians per sample
starting_freq = hz2rad(starting_freq_hz, fs)

predicted_z = torch.exp(1j * starting_freq)
predicted_z.detach_().requires_grad_(True)

print(f"Starting frequency: {rad2hz(predicted_z.angle().abs().item(), fs):.2f}")

target_freq_hz = torch.rand(1) * 0.05 * nyquist_freq
# target_freq_hz = torch.tensor([32.0])
# Convert target frequency to radians per sample
target_freq = hz2rad(target_freq_hz, fs)

# Random target amplitude
# target_amplitude = torch.rand(1) * 0.8 + 0.2  # between 0.2 and 1.0
target_amplitude = torch.tensor([0.1])

# Generate the target signal
target_signal = target_amplitude * torch.cos(n * target_freq)

criterion = torch.nn.MSELoss()
# optimiser = torch.optim.SGD([predicted_z], lr=3e-4)
optimiser = torch.optim.Adam([predicted_z], lr=1e-5)

for step in range(200000):
    predicted_signal = complex_oscillator(predicted_z, N=N, reduce=True)
    loss = criterion(predicted_signal, target_signal)

    optimiser.zero_grad()
    loss.backward()
    predicted_z.grad = predicted_z.grad / predicted_z.grad.abs()
    optimiser.step()

    if (step + 1) % 1000 == 0:
        print(f"--- Step: {step + 1} ---")
        print(f"Predicted frequency: {rad2hz(predicted_z.angle().abs().item(), fs):.2f}")
        print(f"Target frequency: {rad2hz(target_freq.item(), fs):.2f}")
        print(f"Predicted amplitude: {predicted_signal.abs().mean().item():.2f}")
        print(f"Target amplitude: {target_amplitude.item():.2f}")

display(Audio(predicted_signal.cpu().detach().numpy(), rate=fs))
display(Audio(target_signal.cpu().detach().numpy(), rate=fs))